# RMIT Hackathon 2026 — Challenge 1

## Chosen model: `FacebookAI/xlm-roberta-large`

Single fine-tuned model (one approved backbone, one set of weights produces the submission).

**Why this model:** the host's FAQ names it the strongest encoder on the approved list; it is pretrained on 100 languages incl. pt/fr/it,
and at 560M parameters it still trains comfortably on Kaggle's 2 x T4. The safety-tuned DeBERTa checkpoints are English-only,
and the 2B/7B LLMs need LoRA runs that are too slow and fragile for the time left.

**Recipe**
- class-weighted cross-entropy (softened "balanced" weights) for the 62/31/7 class split
- layer-wise LR decay + higher LR for the new classifier head (stabilises XLM-R large)
- head+tail truncation to 256 tokens (injections often sit at the end of long prompts)
- length-bucketed batches + fp16 + both T4s via DataParallel for speed
- **Stage A:** train on 85%, pick the best checkpoint by validation macro-F1 (evaluated twice per epoch), tune per-class logit offsets, write a first `submission.csv`
- **Stage B:** retrain on 100% of `train.csv` for the best epoch count found in stage A and overwrite `submission.csv` (kept only if it passes a sanity check)
- automatic restart with a new seed if XLM-R large collapses to a single class

**Kaggle setup:** add the competition data + model `aisuko/xlm-roberta-large` (Transformers) as inputs, accelerator **GPU T4 x2**,
name the notebook `FPTU_SGS_Rock-D-Xebec_C1`, then *Save Version → Save & Run All*.

In [ ]:
# ==== CHOSEN MODEL: FacebookAI/xlm-roberta-large ====
MODEL_KEY = "xlm-roberta-large"
HF_ID = "FacebookAI/xlm-roberta-large"

MAX_LEN = 256
HEAD_FRAC = 0.5          # share of the token budget kept from the start of a long prompt
BATCH_SIZE = 16          # per optimiser step, split across GPUs on 2 x T4
EPOCHS = 3
EVALS_PER_EPOCH = 2
LR = 1e-5                # top encoder layer
LLRD = 0.95              # each lower layer gets 0.95x the LR of the layer above
HEAD_LR_MULT = 5.0       # freshly initialised classifier head learns faster
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
CLASS_WEIGHT_POWER = 0.5 # 1.0 = sklearn "balanced", 0.5 = softened, 0.0 = no weighting
VAL_FRAC = 0.15
TUNE_BIAS = True         # per-class logit offsets tuned on validation for macro-F1
FULL_FIT = True          # stage B: retrain on 100% of train.csv (roughly doubles runtime)
COLLAPSE_F1 = 0.5        # val macro-F1 below this after one epoch = collapsed run, restart
MAX_ATTEMPTS = 3
SEED = 42

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

In [ ]:
import gc, math, os, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

transformers.logging.set_verbosity_error()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
N_GPU = torch.cuda.device_count()
T_START = time.time()
print("transformers", transformers.__version__, "| torch", torch.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(N_GPU)])

In [ ]:
INPUT = Path("/kaggle/input")

def find_data_file(name):
    hits = sorted(p for p in INPUT.rglob(name) if "rmit" in str(p).lower()) or sorted(INPUT.rglob(name))
    assert hits, f"{name} not found under /kaggle/input — add the competition data as input"
    return hits[0]

def find_model_dir(key):
    hits = sorted(p.parent for p in INPUT.rglob("config.json") if key in str(p).lower())
    if hits:
        return str(hits[0])
    print(f"[warn] no local copy of {key} under /kaggle/input, falling back to the HF hub (needs Internet on)")
    return HF_ID

TRAIN_CSV = find_data_file("train.csv")
TEST_CSV = find_data_file("test.csv")
MODEL_DIR = find_model_dir(MODEL_KEY)
print("train:", TRAIN_CSV, "\ntest: ", TEST_CSV, "\nmodel:", MODEL_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["text"] = train_df["text"].fillna("").astype(str)
test_df["text"] = test_df["text"].fillna("").astype(str)
train_df["y"] = train_df["label"].map(LABEL2ID)
assert train_df["y"].notna().all(), train_df.loc[train_df["y"].isna(), "label"].unique()
train_df["y"] = train_df["y"].astype(int)

print(train_df.shape, test_df.shape)
display(pd.crosstab(train_df["language"], train_df["label"], margins=True))

In [ ]:
# stratify on label x language so every language keeps its class mix in validation
strat = train_df["label"] + "_" + train_df["language"]
tr_df, va_df = train_test_split(train_df, test_size=VAL_FRAC, stratify=strat, random_state=SEED)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)
y_tr, y_va = tr_df["y"].values, va_df["y"].values
print("train", len(tr_df), "| val", len(va_df))
print(va_df["label"].value_counts())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# work out which special tokens this tokenizer wraps around a sequence
_full = tokenizer("hello")["input_ids"]
_core = tokenizer("hello", add_special_tokens=False)["input_ids"]
_i = next(i for i in range(len(_full)) if _full[i:i + len(_core)] == _core)
PREFIX, SUFFIX = _full[:_i], _full[_i + len(_core):]
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
print("prefix", PREFIX, "suffix", SUFFIX, "pad", PAD_ID)

def encode(texts):
    budget = MAX_LEN - len(PREFIX) - len(SUFFIX)
    head = int(budget * HEAD_FRAC)
    ids = tokenizer(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    out = []
    for x in ids:
        if len(x) > budget:
            x = x[:head] + x[len(x) - (budget - head):]
        out.append(PREFIX + x + SUFFIX)
    return out

t0 = time.time()
tr_ids, va_ids, te_ids = encode(tr_df["text"]), encode(va_df["text"]), encode(test_df["text"])
lens = np.array([len(x) for x in tr_ids])
print(f"tokenized in {time.time() - t0:.0f}s | median len {np.median(lens):.0f} | "
      f"truncated {np.mean(lens >= MAX_LEN):.1%}")

In [ ]:
class PromptDataset(Dataset):
    def __init__(self, ids, labels=None):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.ids[i], (-1 if self.labels is None else int(self.labels[i]))

def collate(batch):
    n = max(len(x) for x, _ in batch)
    input_ids = torch.full((len(batch), n), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, (x, _) in enumerate(batch):
        input_ids[i, :len(x)] = torch.tensor(x)
        mask[i, :len(x)] = 1
    return input_ids, mask, torch.tensor([y for _, y in batch])

def smart_batches(lengths, rng, chunk_batches=50):
    # shuffle, sort by length inside chunks of 50 batches, then shuffle the batches:
    # keeps randomness but cuts padding a lot (most prompts are short, a few are huge)
    idx = rng.permutation(len(lengths))
    chunk = BATCH_SIZE * chunk_batches
    batches = []
    for s in range(0, len(idx), chunk):
        c = idx[s:s + chunk]
        c = c[np.argsort(lengths[c], kind="stable")]
        batches += [c[i:i + BATCH_SIZE].tolist() for i in range(0, len(c), BATCH_SIZE)]
    rng.shuffle(batches)
    return batches

@torch.no_grad()
def predict_logits(net, ids):
    net.eval()
    order = np.argsort([len(x) for x in ids])
    loader = DataLoader(PromptDataset([ids[i] for i in order]), batch_size=BATCH_SIZE * 4,
                        shuffle=False, collate_fn=collate)
    chunks = []
    for input_ids, mask, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = net(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        chunks.append(logits.float().cpu().numpy())
    out = np.empty((len(ids), len(LABELS)), dtype=np.float32)
    out[order] = np.concatenate(chunks)
    return out

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

In [ ]:
NO_DECAY = ("bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight")

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID,
    ).to(DEVICE)

def param_groups(model):
    # layer-wise LR decay: top layer gets LR, each layer below 0.95x, embeddings lowest, head LR * HEAD_LR_MULT
    n_layers = model.config.num_hidden_layers
    groups = {}
    for name, p in model.named_parameters():
        m = re.search(r"\.layers?\.(\d+)\.", name)
        if m:
            lr = LR * LLRD ** (n_layers - 1 - int(m.group(1)))
        elif "embeddings" in name:
            lr = LR * LLRD ** n_layers
        else:
            lr = LR * HEAD_LR_MULT
        wd = 0.0 if any(k in name for k in NO_DECAY) else WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()]

def train_model(ids, y, n_epochs, seed, va_ids=None, va_y=None):
    """Fine-tune a fresh model. With validation data, keeps the best checkpoint and stops early on collapse."""
    seed_everything(seed)
    model = build_model()
    net = nn.DataParallel(model) if N_GPU > 1 else model

    counts = np.bincount(y, minlength=len(LABELS))
    w = (len(y) / (len(LABELS) * counts)) ** CLASS_WEIGHT_POWER
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE))

    rng = np.random.default_rng(seed)
    lengths = np.array([len(x) for x in ids])
    steps_per_epoch = len(smart_batches(lengths, rng))
    total_steps = max(1, round(steps_per_epoch * n_epochs))
    optimizer = torch.optim.AdamW(param_groups(model), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    eval_every = max(1, steps_per_epoch // EVALS_PER_EPOCH)
    print(f"seed {seed} | {len(ids)} rows | {n_epochs} epochs = {total_steps} steps | class weights {np.round(w / w.mean(), 3).tolist()}")

    ds = PromptDataset(ids, y)
    info = {"best_f1": -1.0, "best_epoch": n_epochs, "val_logits": None, "collapsed": False}
    best_state, step, t0 = None, 0, time.time()
    while step < total_steps:
        net.train()
        running, n_seen = 0.0, 0
        for input_ids, mask, yb in DataLoader(ds, batch_sampler=smart_batches(lengths, rng), collate_fn=collate):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = net(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
            loss = loss_fn(logits.float(), yb.to(DEVICE))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            running, n_seen = running + loss.item(), n_seen + 1
            if step % 100 == 0:
                print(f"  step {step}/{total_steps} loss {running / n_seen:.4f} ({time.time() - t0:.0f}s)")

            done = step >= total_steps
            if va_ids is not None and (step % eval_every == 0 or done):
                val_logits = predict_logits(net, va_ids)
                f1 = macro_f1(va_y, val_logits.argmax(1))
                ep = step / steps_per_epoch
                print(f"  epoch {ep:.2f}: val macro-F1 {f1:.4f} ({time.time() - t0:.0f}s)")
                if f1 > info["best_f1"]:
                    info.update(best_f1=f1, best_epoch=ep, val_logits=val_logits)
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if ep >= 0.99 and info["best_f1"] < COLLAPSE_F1:
                    info["collapsed"] = True
                    return model, info
                net.train()
            if done:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"done in {time.time() - t0:.0f}s")
    return model, info

## Stage A — train with validation

In [ ]:
for attempt in range(MAX_ATTEMPTS):
    stage_a_seed = SEED + attempt
    model, info = train_model(tr_ids, y_tr, EPOCHS, stage_a_seed, va_ids, y_va)
    if not info["collapsed"] or attempt == MAX_ATTEMPTS - 1:
        break
    print(f"[attempt {attempt + 1}] collapsed (val macro-F1 {info['best_f1']:.3f}) — restarting with another seed")
    del model; free_gpu()

if info["collapsed"]:
    print("[warn] every attempt collapsed — try a lower LR")
val_logits = info["val_logits"]
print(f"stage A: best val macro-F1 {info['best_f1']:.4f} at epoch {info['best_epoch']:.2f}")

In [ ]:
va_pred = val_logits.argmax(1)
print(classification_report(y_va, va_pred, target_names=LABELS, digits=4, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_va, va_pred), index=LABELS, columns=LABELS))
print("\nper-language macro-F1:")
for lang, g in va_df.groupby("language"):
    print(f"  {lang}: {macro_f1(g['y'], va_pred[g.index]):.4f}  (n={len(g)})")

In [ ]:
# Per-class logit offsets tuned on validation; only kept if they clearly help,
# since ~150 PROMPT_INJECTION rows in val is not much to tune on.
bias = np.zeros(len(LABELS), dtype=np.float32)
if TUNE_BIAS:
    grid = np.arange(-2.0, 2.01, 0.25)
    best_b, best_b_f1 = bias.copy(), info["best_f1"]
    for bj in grid:
        for bp in grid:
            b = np.array([0.0, bj, bp], dtype=np.float32)
            f = macro_f1(y_va, (val_logits + b).argmax(1))
            if f > best_b_f1 + 1e-9:
                best_b, best_b_f1 = b, f
    if best_b_f1 - info["best_f1"] >= 0.002:
        bias = best_b
print(f"bias {dict(zip(LABELS, bias.tolist()))} | val macro-F1 {info['best_f1']:.4f} -> "
      f"{macro_f1(y_va, (val_logits + bias).argmax(1)):.4f}")

In [ ]:
def write_submission(logits, source):
    pred = (logits + bias).argmax(1)
    sub = pd.DataFrame({"id": test_df["id"], "label": [LABELS[i] for i in pred]})
    assert len(sub) == len(test_df) == 6000, len(sub)
    assert sub["id"].is_unique and set(sub["label"]) <= set(LABELS)
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print(f"submission.csv written from {source}:")
    print(sub["label"].value_counts(normalize=True).round(3).to_dict())
    return sub

# write a valid submission now so there is one even if stage B fails or runs out of time
test_logits_a = predict_logits(model, te_ids)
np.save("/kaggle/working/test_logits_stageA.npy", test_logits_a)
sub = write_submission(test_logits_a, "stage A")
print(f"elapsed {(time.time() - T_START) / 60:.1f} min")

## Stage B — retrain on 100% of train.csv
Same recipe and seed, for the best epoch count from stage A (rounded up to a half epoch).
Sanity check: the validation rows are now part of training, so the model must score at least as well on them as stage A did; otherwise the stage A submission is kept.

In [ ]:
if FULL_FIT and not info["collapsed"]:
    full_epochs = min(EPOCHS, max(1.0, math.ceil(info["best_epoch"] * 2) / 2))
    del model; free_gpu()
    model, _ = train_model(tr_ids + va_ids, np.concatenate([y_tr, y_va]), full_epochs, stage_a_seed)
    seen_f1 = macro_f1(y_va, predict_logits(model, va_ids).argmax(1))
    print(f"stage B: macro-F1 on (now seen) val rows {seen_f1:.4f}")
    if seen_f1 >= min(0.9, info["best_f1"]):
        test_logits_b = predict_logits(model, te_ids)
        np.save("/kaggle/working/test_logits_stageB.npy", test_logits_b)
        agree = np.mean(test_logits_b.argmax(1) == test_logits_a.argmax(1))
        print(f"stage A / B agree on {agree:.1%} of test rows")
        sub = write_submission(test_logits_b, "stage B (full fit)")
    else:
        print("[warn] stage B looks broken — keeping the stage A submission")
else:
    print("stage B skipped — submission.csv is from stage A")
print(f"total elapsed {(time.time() - T_START) / 60:.1f} min")
sub.head()